# Sentiment Analysis of Touch 'n Go eWallet App Reviews
## Step 1: Scraping English reviews from Google Play Store

- The latest 600 five-star reviews and the latest 600 one-star reviews
- English only

In [1]:
%pip install -q google-play-scraper langdetect pandas openpyxl


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Settings

In [2]:
import pandas as pd
from google_play_scraper import reviews, Sort
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0           # same language result every run

APP_ID = 'my.com.tngdigital.ewallet'
TARGET = 600                       # reviews needed per rating

### Check if a review is English

In [3]:
def is_english(text):
    try:
        return len(text.split()) >= 3 and detect(text) == 'en'
    except Exception:
        return False

### Scrape one rating
Gets reviews from newest to oldest and keeps the English ones until it has 600.

In [4]:
def scrape(star):
    kept = []
    token = None
    while len(kept) < TARGET:
        result, token = reviews(APP_ID, lang='en', country='my', sort=Sort.NEWEST,
                                count=200, filter_score_with=star, continuation_token=token)
        for r in result:
            if is_english(r['content']):
                kept.append(r)
        if not result or token is None:
            break   # no more reviews on Google Play
    kept = kept[:TARGET]
    print(f'{star}-star: {len(kept)} English reviews')
    return pd.DataFrame(kept)

In [5]:
five_star = scrape(5)
one_star = scrape(1)

5-star: 600 English reviews
1-star: 600 English reviews


### Build the dataset

In [6]:
df = pd.concat([five_star, one_star])

df = df[['reviewId', 'content', 'score', 'at']]
df.columns = ['review_id', 'review', 'rating', 'review_date']
df = df.reset_index(drop=True)

print(df['rating'].value_counts())
print('5-star dates:', df[df.rating == 5].review_date.min(), 'to', df[df.rating == 5].review_date.max())
print('1-star dates:', df[df.rating == 1].review_date.min(), 'to', df[df.rating == 1].review_date.max())
df.head()

rating
5    600
1    600
Name: count, dtype: int64
5-star dates: 2026-08-01 20:31:29 to 2026-11-14 16:23:47
1-star dates: 2026-05-23 14:47:46 to 2026-11-14 18:46:43


,review_id,review,rating,review_date
0,61c287f4-fac1-477c-bca2-3e1cddcb3cb6,easy to use,5,2026-11-14 16:23:47
1,6f265428-a577-4674-b41f-4ce009a269d5,But quite confuse to used the rewarding points...,5,2026-11-14 11:56:58
2,2eca021b-6638-42de-8003-d1ab6362bd33,easy and very good with a safe app,5,2026-11-14 09:35:31
3,15b5dd7f-7d20-4d01-b6a0-2b000bbb37d6,very nice app,5,2026-11-14 03:57:03
4,b3a841d9-a147-4702-b262-ccdc6c198d3e,Easy to use,5,2026-11-13 20:11:19


### Save as Excel

In [7]:
import os
os.makedirs('data', exist_ok=True)
df.to_excel('data/tng_reviews.xlsx', index=False)
print('Saved to data/tng_reviews.xlsx')

Saved to data/tng_reviews.xlsx
